In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install einops -q

import pathlib
import collections
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import einops

print('TF version:', tf.__version__)
print('Keras version:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TF version: 2.19.0
Keras version: 3.13.2
GPUs: []


In [3]:
DATA_ROOT  = '/content/drive/MyDrive/CCTV_dataset/data'
HEIGHT     = 224
WIDTH      = 224
N_FRAMES   = 10
BATCH_SIZE = 16
EPOCHS     = 50
LR         = 1e-4

# ---- class map: must match your actual folder names exactly ----
# If crash types ARE their own folders (train/head-on/, train/rear-end/, ...)
# use CLASS_MAP as-is.
# If crash types are encoded in the filename instead, see build_clip_paths below.
CLASS_MAP = {
    'head-on':   0,
    'rear-end':  1,
    'sideswipe': 2,
    'single':    3,
    't-bone':    4,
}
CLASS_NAMES = ['head-on', 'rear-end', 'sideswipe', 'single', 't-bone']
N_CLASSES   = len(CLASS_MAP)

In [4]:
def parse_crash_type_from_filename(stem: str) -> str | None:
    """
    Extracts crash type from a filename stem like:
      'Town03_head-on_clear_16_frame_0042'
      'Town05_t-bone_rain_07_frame_0010'

    Strategy: scan every known class name as a substring — avoids
    fragile positional splitting and handles hyphens in type names.
    """
    for class_name in CLASS_MAP:
        if f'_{class_name}_' in stem or stem.startswith(f'{class_name}_'):
            return class_name
    return None


def build_clip_paths(split_dir: str, n_frames: int = N_FRAMES):
    root = pathlib.Path(split_dir)
    all_clip_paths = []
    all_labels     = []

    # Key: (crash_type, video_id) → list of frame paths
    video_frames = collections.defaultdict(list)
    skipped = 0

    for img_path in sorted(root.glob('**/*.jpg')):
        # 1) Try to get class from parent folder name
        class_name = img_path.parent.name if img_path.parent.name in CLASS_MAP else None

        # 2) Fall back to filename parsing
        if class_name is None:
            class_name = parse_crash_type_from_filename(img_path.stem)

        if class_name is None:
            skipped += 1
            if skipped <= 5:  # only print first few warnings
                print(f'WARNING: could not determine crash type for {img_path.name} — skipping.')
            continue

        parts = img_path.stem.split('_frame_')
        if len(parts) != 2:
            skipped += 1
            continue

        video_id = parts[0]
        video_frames[(class_name, video_id)].append(str(img_path))

    if skipped > 5:
        print(f'WARNING: {skipped} total files skipped (showing first 5 above).')

    for (class_name, video_id), frames in video_frames.items():
        label  = CLASS_MAP[class_name]
        frames = sorted(frames)
        for start in range(0, len(frames) - n_frames + 1, n_frames):
            all_clip_paths.append(frames[start : start + n_frames])
            all_labels.append(label)

    return all_clip_paths, all_labels


train_paths, train_labels = build_clip_paths(f'{DATA_ROOT}/train')
val_paths,   val_labels   = build_clip_paths(f'{DATA_ROOT}/val')
test_paths,  test_labels  = build_clip_paths(f'{DATA_ROOT}/test')

for split_name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    print(f'{split_name} clips: {len(labels)}')
    for i, name in enumerate(CLASS_NAMES):
        print(f'  {name}: {labels.count(i)}')

Train clips: 5071
  head-on: 1484
  rear-end: 1687
  sideswipe: 939
  single: 127
  t-bone: 834
Val clips: 566
  head-on: 151
  rear-end: 211
  sideswipe: 103
  single: 17
  t-bone: 84
Test clips: 1396
  head-on: 354
  rear-end: 505
  sideswipe: 247
  single: 35
  t-bone: 255


In [5]:
MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)


def load_frame(path: tf.Tensor) -> tf.Tensor:
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [HEIGHT, WIDTH])
    img = tf.cast(img, tf.float32) / 255.0
    img = (img - MEAN) / STD
    return img


def load_clip(frame_paths: tf.Tensor, label: tf.Tensor):
    frames = tf.map_fn(load_frame, frame_paths, fn_output_signature=tf.float32)
    return frames, label


def augment_clip(frames: tf.Tensor, label: tf.Tensor):
    flip   = tf.random.uniform(()) > 0.5
    frames = tf.cond(flip, lambda: frames[:, :, ::-1, :], lambda: frames)
    frames = tf.image.random_brightness(frames, max_delta=0.1)
    frames = tf.image.random_contrast(frames, lower=0.9, upper=1.1)
    frames = tf.clip_by_value(frames, -3.0, 3.0)
    return frames, label


def make_dataset(clip_paths, labels, training=False, batch_size=BATCH_SIZE):
    path_tensor  = tf.ragged.constant(clip_paths).to_tensor()
    label_tensor = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((path_tensor, label_tensor))
    if training:
        ds = ds.shuffle(buffer_size=min(len(labels), 1000), reshuffle_each_iteration=True)
    ds = ds.map(load_clip, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment_clip, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds   = make_dataset(val_paths,   val_labels,   training=False)
test_ds  = make_dataset(test_paths,  test_labels,  training=False)

for clips, lbls in train_ds.take(1):
    print('Clip batch shape: ', clips.shape)   # (BATCH_SIZE, N_FRAMES, H, W, 3)
    print('Label batch shape:', lbls.shape)    # (BATCH_SIZE,)

Clip batch shape:  (16, 10, 224, 224, 3)
Label batch shape: (16,)


In [6]:
class_counts = collections.Counter(train_labels)
n_total      = len(train_labels)

class_weight = {
    i: n_total / (N_CLASSES * class_counts[i]) if class_counts[i] > 0 else 1.0
    for i in range(N_CLASSES)
}

print('Class weights:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {class_weight[i]:.3f}')

Class weights:
  head-on: 0.683
  rear-end: 0.601
  sideswipe: 1.080
  single: 7.986
  t-bone: 1.216


In [7]:
class Conv2Plus1D(keras.layers.Layer):
    def __init__(self, filters, kernel_size, padding='same'):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size, kernel_size)
        self.seq = keras.Sequential([
            layers.Conv3D(filters, kernel_size=(1, kernel_size[1], kernel_size[2]), padding=padding),
            layers.Conv3D(filters, kernel_size=(kernel_size[0], 1, 1),             padding=padding),
        ])

    def call(self, x):
        return self.seq(x)


class ResidualMain(keras.layers.Layer):
    def __init__(self, filters, kernel_size):
        super().__init__()
        self.seq = keras.Sequential([
            Conv2Plus1D(filters, kernel_size, padding='same'),
            layers.LayerNormalization(),
            layers.ReLU(),
            Conv2Plus1D(filters, kernel_size, padding='same'),
            layers.LayerNormalization(),
        ])

    def call(self, x):
        return self.seq(x)


class Project(keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.seq = keras.Sequential([
            layers.Dense(units),
            layers.LayerNormalization(),
        ])

    def call(self, x):
        return self.seq(x)


def add_residual_block(x, filters, kernel_size):
    out = ResidualMain(filters, kernel_size)(x)
    res = x if out.shape[-1] == x.shape[-1] else Project(out.shape[-1])(x)
    return layers.add([res, out])


class ResizeVideo(keras.layers.Layer):
    def __init__(self, height, width):
        super().__init__()
        self.height = height
        self.width  = width
        self.resize = layers.Resizing(height, width)

    def call(self, video):
        shape  = einops.parse_shape(video, 'b t h w c')
        images = einops.rearrange(video, 'b t h w c -> (b t) h w c')
        images = self.resize(images)
        return einops.rearrange(images, '(b t) h w c -> b t h w c', t=shape['t'])


def build_model(n_frames=N_FRAMES, height=HEIGHT, width=WIDTH, num_classes=N_CLASSES):
    inp = layers.Input(shape=(n_frames, height, width, 3))
    x   = inp

    x = Conv2Plus1D(16, (3, 7, 7), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = ResizeVideo(height // 2, width // 2)(x)

    x = add_residual_block(x, 16,  (3, 3, 3))
    x = ResizeVideo(height // 4, width // 4)(x)

    x = add_residual_block(x, 32,  (3, 3, 3))
    x = ResizeVideo(height // 8, width // 8)(x)

    x = add_residual_block(x, 64,  (3, 3, 3))
    x = ResizeVideo(height // 16, width // 16)(x)

    x = add_residual_block(x, 128, (3, 3, 3))

    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(num_classes)(x)  # logits — 5 classes

    return keras.Model(inp, x)


model = build_model()
model.summary()

Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_plus1d        │ (None, 10, 224,   │      3,152 │ input_layer[0][0] │
│ (Conv2Plus1D)       │ 224, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 10, 224,   │         64 │ conv2_plus1d[0][… │
│ (BatchNormalizatio… │ 224, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 10, 224,   │          0 │ batch_normalizat… │
│                     │ 224, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resize_video        │ (None, 10, 112,   │          0 │ re_lu[0][0]       │
│ (ResizeVideo)       │ 112, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_main       │ (None, 10, 112,   │      6,272 │ resize_video[0][… │
│ (ResidualMain)      │ 112, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 112,   │          0 │ resize_video[0][… │
│                     │ 112, 16)          │            │ residual_main[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resize_video_1      │ (None, 10, 56,    │          0 │ add[0][0]         │
│ (ResizeVideo)       │ 56, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ project (Project)   │ (None, 10, 56,    │        608 │ resize_video_1[0… │
│                     │ 56, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_main_1     │ (None, 10, 56,    │     20,224 │ resize_video_1[0… │
│ (ResidualMain)      │ 56, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 56,    │          0 │ project[0][0],    │
│                     │ 56, 32)           │            │ residual_main_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resize_video_2      │ (None, 10, 28,    │          0 │ add_1[0][0]       │
│ (ResizeVideo)       │ 28, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ project_1 (Project) │ (None, 10, 28,    │      2,240 │ resize_video_2[0… │
│                     │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_main_2     │ (None, 10, 28,    │     80,384 │ resize_video_2[0… │
│ (ResidualMain)      │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 28,    │          0 │ project_1[0][0],  │
│                     │ 28, 64)           │            │ residual_main_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resize_video_3      │ (None, 10, 14,    │          0 │ add_2[0][0]       │
│ (ResizeVideo)       │ 14, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ project_2 (Project) │ (None, 10, 14,    │      8,576 │ resize_video_3[0

 Total params: 442,677 (1.69 MB)

 Trainable params: 442,645 (1.69 MB)

 Non-trainable params: 32 (128.00 B)

In [8]:
model.compile(
    loss      = keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer = keras.optimizers.Adam(learning_rate=LR),
    metrics   = [
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
    ]
)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        mode='max',
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    epochs          = EPOCHS,
    validation_data = val_ds,
    class_weight    = class_weight,
    callbacks       = callbacks,
)

Epoch 1/50
  5/317 ━━━━━━━━━━━━━━━━━━━━ 1:20:27 15s/step - accuracy: 0.2946 - loss: 1.7263

In [ ]:
def plot_history(history):
    metrics = ['loss', 'accuracy']
    fig, axes = plt.subplots(1, len(metrics), figsize=(10, 4))
    for ax, metric in zip(axes, metrics):
        ax.plot(history.history[metric],          label='train')
        ax.plot(history.history[f'val_{metric}'], label='val')
        ax.set_title(metric.replace('_', ' ').capitalize())
        ax.set_xlabel('Epoch')
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_history(history)

In [ ]:
results = model.evaluate(test_ds, return_dict=True)
print('\nTest results:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')